# AuraScribble Retrain (ONNX-Only Dev)

Notebook for iterative retraining + ONNX export + Firebase OTA upload.
This flow is tuned for the issues we hit today: dynamic-shape export stability, vocab sync, and short-sequence ONNX runtime checks.

In [ ]:
import os
from pathlib import Path

ROOT = Path('/content/handwriting-model')
assert ROOT.exists(), f'Missing repo path: {ROOT}'
os.chdir(ROOT)
print('cwd =', Path.cwd())
!python -V
!pip -q install -r requirements.txt

## 1) Build balanced split (real data only)

If `data/processed/all.jsonl` does not exist yet, build it from your normal data prep pipeline first.

In [ ]:
!python -u src/split_data.py \
  --input data/processed/all.jsonl \
  --train-out data/processed/train.jsonl \
  --val-out data/processed/val.jsonl \
  --test-out data/processed/test.jsonl \
  --val-ratio 0.1 \
  --test-ratio 0.1 \
  --seed 42 \
  --balance-modes

In [ ]:
!python - <<'PY'
import json
from pathlib import Path

for name in ['train', 'val', 'test']:
    p = Path(f'data/processed/{name}.jsonl')
    n = sum(1 for _ in p.open('r', encoding='utf-8')) if p.exists() else 0
    print(name, n)
PY

## 2) Retrain

Use the config you already stabilized (`configs/train.yaml`), or replace with your latest tuned config.

In [ ]:
!python -u src/train.py --config configs/train.yaml

## 3) Export ONNX + short-sequence smoke test

This uses the updated export flow (no `jit.trace`) and validates runtime at short input length (38).

In [ ]:
!python -u src/export_onnx.py \
  --config configs/train.yaml \
  --checkpoint output/checkpoint_best.pt \
  --trace-time 128 \
  --trace-tokens 128 \
  --smoke-time 38 \
  --summary output/export_summary.json

In [ ]:
!python - <<'PY'
import json
from pathlib import Path

summary = json.loads(Path('output/export_summary.json').read_text(encoding='utf-8'))
print('status:', summary.get('status'))
print('vocab_size:', summary.get('vocab_size'))
print('onnx:', summary.get('onnx_model'))
print('quantized:', summary.get('quantized_onnx_model'))
print('vocab_for_ota:', summary.get('vocab_for_ota'))
PY

## 4) Hard-check local vocab count (must include space token)

Use raw line count (no `.strip()` filtering), otherwise a `' '` token appears as missing.

In [ ]:
!python - <<'PY'
from pathlib import Path
p = Path('output/vocab.from_checkpoint.txt')
raw = p.read_text(encoding='utf-8').splitlines()
print('raw_count:', len(raw))
print('has_space_token:', ' ' in raw)
print('space_idx:', raw.index(' ') if ' ' in raw else None)
PY

## 5) Upload ONNX + checkpoint vocab to Firebase OTA paths

In [ ]:
!python -u src/upload_firebase.py \
  --local output/model.onnx \
  --vocab output/vocab.from_checkpoint.txt \
  --vocab-remote models/latest_vocab.txt

## 6) Verify remote vocab is exact (raw count)

No `strip()` in counting.

In [ ]:
!python - <<'PY'
from pathlib import Path
from src.upload_firebase import _resolve_credentials, DEFAULT_BUCKET
from google.cloud import storage

creds = _resolve_credentials(None)
if creds is None:
    raise RuntimeError('Missing Firebase credentials (service account)')

client = storage.Client(credentials=creds, project=creds.project_id)
out = Path('/tmp/latest_vocab.txt')
client.bucket(DEFAULT_BUCKET).blob('models/latest_vocab.txt').download_to_filename(str(out))
raw = out.read_text(encoding='utf-8').splitlines()
print('remote_raw_count:', len(raw))
print('remote_has_space_token:', ' ' in raw)
print('remote_space_idx:', raw.index(' ') if ' ' in raw else None)
PY

## 7) Development loop checklist (Android)

- Use debug build with `ONNX_ONLY_DEBUG=true` (ML Kit fallback disabled).
- Clear app data between OTA iterations.
- Confirm logs: `Vocab: source=ota size=...`, `ONNX model source: ota`.
- Track per-sample outputs (word/sentence/math) and feed corrections back into the next retrain.